# Casino Machine Metrics ETL

This pipeline builds the **`default.machine_metrics_clean`** Delta table for The Gilded Ace floor analysis.

**Pipeline steps**

1. **Ingest** — read `prepared_daily_view.parquet` (machine × day) and `calendar_day.parquet` (day) from the cloned `dashboard-royale` repo in the workspace.
   > Note: the DBFS public root (`/FileStore/tables`) is disabled on this workspace, so the same files are read from the workspace copy instead.
2. **Join & clean** — inner join on `gaming_date`, keep only rows where `active == True`.
3. **Aggregate** — group by `machine_id` and compute `total_house_win`, `expected_win`, `adjusted_hold`, and `availability_ratio`, guarding divisions against zero denominators.
4. **Save** — write to the Delta table `default.machine_metrics_clean` in `overwrite` mode.

**Run order:** top to bottom. The final cell validates the output table.

In [0]:
from pyspark.sql import functions as F

# Input Parquet files (workspace copy of the dashboard-royale repo; DBFS /FileStore is disabled here)
DAILY_VIEW_PATH = "/Workspace/Users/saharshgurram@tamu.edu/dashboard-royale/data/1_prepared_views/prepared_daily_view.parquet"
CALENDAR_DAY_PATH = "/Workspace/Users/saharshgurram@tamu.edu/dashboard-royale/data/2_machine_decision/calendar_day.parquet"

# 1 · Ingest: read both Parquet sources
prepared_daily_view = spark.read.parquet(DAILY_VIEW_PATH)
calendar_day = spark.read.parquet(CALENDAR_DAY_PATH)

print(f"prepared_daily_view: {prepared_daily_view.count()} rows")
print(f"calendar_day:        {calendar_day.count()} rows")
display(prepared_daily_view.limit(5))

prepared_daily_view: 56110 rows
calendar_day:        181 rows


machine_id,gaming_date,month,local_weekday,is_weekend,federal_holiday,events_active,zone_id,zone_name,bank_id,location_id,config_id,theme,configured_rtp,volatility_class,manufacturer,lease_class,active,meter_status,available_hours,occupied_hours,availability_ratio,occupancy_ratio,paid_spins,coin_in_usd,payout_usd,house_win_usd,theo_usd,observed_hold,theo_hold,luck_residual_usd,carded_coin_in_usd,carded_share,faults_opened,zone_entries_observed,zone_counter_coverage_hours,lease_cost_usd,service_cost_usd,power_cost_usd,repair_cost_usd,contribution_after_supplied_costs_usd,lag_coin_in_7d_mean_usd,lag_downtime_7d_hours,lag_house_win_28d_usd
M-1001,2025-01-01,2025-01,Wed,false,New Year's Day,null,Z-ENT,Main Entrance,B-01,L-01-6,G-PEN89,Prairie Thunder,0.89000000,medium,Halcyon,leased_fixed,true,reported,24.0,10.548055555555555,1.0,0.43950231481481483,6253,5268.40,4344.40,924.00,579.5240,0.17538531622503986,0.11,344.4760,4388.80,0.8330422898792802,0,1999,23.0,45.00,6.00,6.00,0.00,867.00,null,null,null
M-1001,2025-01-02,2025-01,Thu,false,null,null,Z-ENT,Main Entrance,B-01,L-01-6,G-PEN89,Prairie Thunder,0.89000000,medium,Halcyon,leased_fixed,true,reported,24.0,10.231111111111112,1.0,0.4262962962962963,6275,4314.80,3569.20,745.60,474.6280,0.17280059330675812,0.11,270.9720,2938.40,0.6810049133215909,0,2530,24.0,45.00,6.00,6.00,0.00,688.60,5268.4,0.0,924.0
M-1001,2025-01-03,2025-01,Fri,false,null,null,Z-ENT,Main Entrance,B-01,L-01-6,G-PEN89,Prairie Thunder,0.89000000,medium,Halcyon,leased_fixed,true,reported,24.0,14.368333333333334,1.0,0.5986805555555555,9404,4945.60,4198.60,747.00,544.0160,0.15104335166612748,0.11,202.9840,3617.20,0.7313976059527661,0,3145,24.0,45.00,6.00,6.00,0.00,690.00,4791.6,0.0,1669.6
M-1001,2025-01-04,2025-01,Sat,true,null,null,Z-ENT,Main Entrance,B-01,L-01-6,G-PEN89,Prairie Thunder,0.89000000,medium,Halcyon,leased_fixed,true,reported,24.0,11.64638888888889,1.0,0.4852662037037037,7313,6137.20,5172.40,964.80,675.0920,0.1572052401746725,0.11,289.7080,4367.20,0.7115948641074106,0,3560,24.0,45.00,6.00,6.00,0.00,907.80,4842.933333333333,0.0,2416.6
M-1001,2025-01-05,2025-01,Sun,true,null,null,Z-ENT,Main Entrance,B-01,L-01-6,G-PEN89,Prairie Thunder,0.89000000,medium,Halcyon,leased_fixed,true,reported,24.0,13.17361111111111,1.0,0.5489004629629629,8381,6702.80,5506.60,1196.20,737.3080,0.1784627319926001,0.11,458.8920,5863.20,0.8747389150802649,0,2865,24.0,45.00,6.00,6.00,0.00,1139.20,5166.5,0.0,3381.3999999999996


In [0]:
# 2 · Join & Clean: daily view ⨝ calendar on gaming_date, keep only active machines.
# Only the join key + scheduled_seconds are taken from the calendar to avoid
# ambiguous duplicate columns (month, local_weekday, is_weekend, federal_holiday).
active_machine_days = (
    prepared_daily_view
    .join(
        calendar_day.select("gaming_date", "scheduled_seconds"),
        on="gaming_date",
        how="inner",
    )
    .filter(F.col("active") == True)
)

print(f"active machine-days kept: {active_machine_days.count()}")

active machine-days kept: 55092


In [0]:
# 3 · Aggregate: one row per machine.
# Ratios are computed from summed numerators/denominators, with when().otherwise()
# guards so zero denominators yield NULL instead of a division-by-zero error.
sums = (
    active_machine_days
    .groupBy("machine_id")
    .agg(
        F.sum("house_win_usd").alias("total_house_win"),
        F.sum("theo_usd").alias("expected_win"),
        F.sum("coin_in_usd").alias("total_coin_in"),
        F.sum("available_hours").alias("total_available_hours"),
        F.sum("scheduled_seconds").alias("total_scheduled_seconds"),
    )
)

machine_metrics = (
    sums
    .withColumn(
        "adjusted_hold",
        F.when(F.col("total_coin_in") != 0,
               F.col("total_house_win") / F.col("total_coin_in"))
         .otherwise(None),
    )
    .withColumn(
        "availability_ratio",
        F.when(F.col("total_scheduled_seconds") != 0,
               F.col("total_available_hours") / (F.col("total_scheduled_seconds") / 3600.0))
         .otherwise(None),
    )
    # Keep only the specified output columns
    .select(
        "machine_id",
        "total_house_win",
        "expected_win",
        "adjusted_hold",
        "availability_ratio",
    )
)

display(machine_metrics.orderBy("total_house_win"))

machine_id,total_house_win,expected_win,adjusted_hold,availability_ratio
M-5076,3573.25,7478.7000,0.0238895129,0.9948979591836735
M-1158,15796.25,51961.0000,0.0304002040,0.9993058433750351
M-1120,16355.00,18910.9250,0.0432421999,0.9946838052134863
M-1118,20050.00,25329.0000,0.0791582771,0.9950481902356902
M-1238,24932.75,33139.3125,0.0376180858,0.99742196894108
M-1181,26977.00,32516.1600,0.0663719209,0.9978396909458387
M-1035,30303.50,30894.6000,0.0490433603,0.9991679458643539
M-1179,30393.75,33784.7000,0.0719704482,0.9959075907590761
M-1017,31939.50,29525.6375,0.0540877398,0.9976078363650318
M-1265,32071.00,32208.8800,0.0796575354,0.9981456110727354


In [0]:
OUTPUT_TABLE = "default.machine_metrics_clean"

# 4 · Save as a Delta table (overwrite mode)
machine_metrics.write.format("delta").mode("overwrite").saveAsTable(OUTPUT_TABLE)

# Validate: read the table back and sanity-check the result
result = spark.table(OUTPUT_TABLE)
row_count = result.count()
null_ratios = result.where(F.col("adjusted_hold").isNull() | F.col("availability_ratio").isNull()).count()

print(f"Table {OUTPUT_TABLE}: {row_count} machines written")
print(f"Rows with NULL adjusted_hold or availability_ratio (zero-denominator guard): {null_ratios}")
print("Bottom 5 machines by total_house_win (the manager's replacement candidates):")
display(result.orderBy("total_house_win").limit(5))

Table default.machine_metrics_clean: 310 machines written
Rows with NULL adjusted_hold or availability_ratio (zero-denominator guard): 0
Bottom 5 machines by total_house_win (the manager's replacement candidates):


machine_id,total_house_win,expected_win,adjusted_hold,availability_ratio
M-5076,3573.25,7478.7000,0.0238895129,0.9948979591836735
M-1158,15796.25,51961.0000,0.0304002040,0.9993058433750351
M-1120,16355.00,18910.9250,0.0432421999,0.9946838052134863
M-1118,20050.00,25329.0000,0.0791582771,0.9950481902356902
M-1238,24932.75,33139.3125,0.0376180858,0.99742196894108


In [0]:
OUTPUT_TABLE = "default.machine_metrics_clean"
EXPORT_PATH = "/Workspace/Users/saharshgurram@tamu.edu/dashboard-royale/data/machine_metrics_clean.csv"

# 1 · Load the clean Delta table with Spark
clean_metrics = spark.table(OUTPUT_TABLE)

# 2 · Convert to a Pandas DataFrame (small table: 310 machines, safe to collect)
clean_metrics_pdf = clean_metrics.toPandas()

# 3 · Save as a CSV file in the workspace
clean_metrics_pdf.to_csv(EXPORT_PATH, index=False)

print(f"Success: exported {len(clean_metrics_pdf)} rows to {EXPORT_PATH}")

Success: exported 310 rows to /Workspace/Users/saharshgurram@tamu.edu/dashboard-royale/data/machine_metrics_clean.csv
